# NeuroTrail brain-age training pipeline

This notebook runs the **whole MRI brain-age training pipeline** of the `brainage-accuracy` branch, step by step, and shows the results.
It does not contain its own copy of the training code: every step calls the real script in `models/`, so the notebook and the scripts can never disagree.

**In plain words:** we teach computer models to guess a person's age from a brain MRI, using only healthy people. Then we check, fairly, how far off they are on people they never saw, and whether people with dementia come out looking "older". Finally we combine the models and attach an honest range and a confidence label to each guess.

```
OASIS scans ──► fixed fair groups (splits.json) ──► train each model ──► out-of-fold guesses
                                                     ├ baseline 4 mm CNN
                                                     ├ new 2 mm CNN
                                                     ├ SFCN (pretrained on UK Biobank, fine-tuned)
                                                     └ volume model
        ──► shared scorer ──► calibrated ranges + ensemble ──► comparison report ──► app
```

**How to run:** in VS Code, pick the kernel **`C:\brain-mri\.venv`** (top right, *Select Kernel → Python Environments*), then run cells top to bottom.

**Safe by default:** each training step is skipped when its results already exist (set it to `"force"` to redo it). The long SFCN step is **off** unless you turn it on.

| Step | Approx. time on an RTX 3060 laptop GPU |
|---|---|
| Download data | 10–20 min (once) |
| Baseline 4 mm CNN | ~5 min |
| Volume model | < 1 min |
| 2 mm CNN (5 groups × 3 seeds) | ~1 hour |
| SFCN fine-tuning | ~2.5–3.5 hours with 10–15 stage-2 passes |
| Calibration, ensemble, report | a few minutes (CPU) |

## Settings

`"auto"` = run only if the results are missing · `"force"` = run again (overwrites results) · `"skip"` = don't run.

In [ ]:
RUN = {
    "download": "auto",
    "baseline_4mm": "auto",   # NOTE: "force" also overwrites the app's current model files in models/artifacts/
    "volume": "auto",
    "cnn_2mm": "auto",
    "sfcn_ft": "skip",        # ~3 hours: set to "auto" when the laptop can stay plugged in and awake
    "ensemble": "auto",
}
SFCN_EPOCHS2 = 15   # max full-model passes per group (each pass ~2.5 min on a 6 GB laptop GPU)
CNN_2MM_SEEDS = 3   # copies per group with different random starts

## Setup

Finds the repo, checks the GPU and the data, and defines `run(...)`, which runs a script and streams its log here.

In [ ]:
import json, os, subprocess, sys
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / "models" / "eval_brainage.py").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
RESULTS, MRI_DIR = ROOT / "results", ROOT / "data" / "raw" / "oasis_mri"
PY = sys.executable
ENV = {**os.environ, "PYTHONUTF8": "1"}   # Windows: read files as UTF-8

def run(script, *args):
    """Run a repo script with this notebook's Python and stream its output."""
    cmd = [PY, script, *map(str, args)]
    print("$", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=ROOT, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, encoding="utf-8", errors="replace")
    for line in p.stdout:
        print(line, end="", flush=True)
    if p.wait() != 0:
        raise RuntimeError(f"{script} failed (exit code {p.returncode})")

def should_run(step, *outputs):
    mode = RUN[step]
    if mode == "skip":
        print(f"[{step}] skipped (RUN['{step}'] = 'skip')"); return False
    if mode == "auto" and all((ROOT / o).exists() for o in outputs):
        print(f"[{step}] results already exist, skipping (set RUN['{step}'] = 'force' to redo)"); return False
    return True

import torch
print("repo:", ROOT)
print("python:", PY)
print("torch", torch.__version__, "| GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU only: training will be very slow)")
print("MRI scans found:", len(list(MRI_DIR.glob("**/*.nii*"))), "(expected 436)")

## Step 0 · Download the data

Public data only: CDC NHANES survey files and OASIS-1 MRI from Kaggle (needs `~/.kaggle/kaggle.json`). About 1.6 GB.

In [ ]:
if should_run("download", "data/raw/oasis_mri/oasis_cross-sectional.csv", "data/raw/oasis/oasis_cross-sectional.csv"):
    run("data/download_data.py")

## Step 1 · Fair groups (folds)

The 316 healthy people are split into 5 groups with a similar age spread. Every model learns from 4 groups and is tested on the 5th, so each person is scored by a model that never saw them.
Repeat scans stay with their person; the 100 people with dementia are never used for learning (group −1).
The split is saved once in `results/splits.json` and reused by every model.

In [ ]:
run("models/splits.py")

## Step 2 · Baseline 4 mm CNN (the original model)

Looks at a shrunken 4 mm copy of each scan. One copy per group, 70 passes each.
Writes `results/oof_baseline_4mm.csv` and also refreshes the app's model files in `models/artifacts/`.

In [ ]:
if should_run("baseline_4mm", "results/oof_baseline_4mm.csv"):
    run("models/train_mri_cnn.py")
run("models/eval_brainage.py", "baseline_4mm")

## Step 3 · Volume model

No image model: just three numbers per scan from OASIS (brain volume share, head size, sex). A simple reference point.

In [ ]:
if should_run("volume", "results/oof_volume.csv"):
    run("models/oof_volume_model.py")

## Step 4 · New 2 mm CNN

Looks at the full-detail 2 mm scan (cropped to the brain). Trained 3 times per group from different random starts; their spread tells us how stable the guess is.
Practice scans are nudged a few millimetres and made slightly brighter or darker each time, so the model learns real patterns instead of memorizing pixels.
Each copy stops when its pop-quiz score (a held-back slice of its own training group) stops improving.

In [ ]:
if should_run("cnn_2mm", "results/oof_cnn_2mm.csv"):
    run("models/train_mri_cnn_2mm.py", "--seeds", CNN_2MM_SEEDS)

## Step 5 · SFCN, pretrained on UK Biobank, fine-tuned on OASIS (long)

SFCN (Peng et al. 2021, MIT license) already learned from ~13,000 UK brains aged 44–80. We download its weights (hash-checked), first check it on our scans untouched, then retrain it a little:
stage 1 = only its top layers learn (8 passes), stage 2 = everything learns at a 10× lower rate, up to `SFCN_EPOCHS2` passes with early stopping.
Our 2 mm scans are upsampled to its 1 mm input size; that adds no real detail.

Off by default (`RUN["sfcn_ft"] = "skip"`) because it takes hours on a laptop GPU.

In [ ]:
if RUN["sfcn_ft"] != "skip" and not (RESULTS / "sfcn_pretrained_check.json").exists():
    run("models/finetune_sfcn.py", "--check-pretrained")
if should_run("sfcn_ft", "results/oof_sfcn_ft.csv"):
    run("models/finetune_sfcn.py", "--epochs2", SFCN_EPOCHS2)
if (RESULTS / "sfcn_pretrained_check.json").exists():
    print(json.dumps(json.loads((RESULTS / "sfcn_pretrained_check.json").read_text()), indent=1))

## Step 6 · Ranges, confidence labels and the ensemble

For each finished image model, and for the combination of all of them (`ensemble`, what the app uses):
- **weights:** better models get more say (∝ 1 / error²), learned on the other groups only;
- **range:** models disagreeing + typical error at that predicted age, scaled so 95% ranges really contain the true age ~95% of the time;
- **confidence label:** high / medium / low with reasons (range width, model disagreement, scan quality, age range).

Also an `ensemble_with_volume` for comparison (the app can't use the volume model: uploads don't come with OASIS volume numbers).

In [ ]:
image_models = [m for m in ("baseline_4mm", "cnn_2mm", "sfcn_ft") if (RESULTS / f"oof_{m}.csv").exists()]
print("finished image models:", image_models)
force = RUN["ensemble"] == "force"
if RUN["ensemble"] != "skip":
    for m in image_models:
        if force or not (RESULTS / f"cal_{m}_config.json").exists():
            run("models/ensemble.py", "--name", f"cal_{m}", "--models", m)
    if len(image_models) >= 2:
        stale = force or not (RESULTS / "ensemble_config.json").exists() or \
            json.loads((RESULTS / "ensemble_config.json").read_text())["models"] != image_models
        if stale:
            run("models/ensemble.py", "--name", "ensemble", "--models", *image_models)
        if (RESULTS / "oof_volume.csv").exists() and (stale or not (RESULTS / "ensemble_with_volume_config.json").exists()):
            run("models/ensemble.py", "--name", "ensemble_with_volume", "--models", *image_models, "volume")

## Step 7 · Comparison report

`results/comparison.md` is generated from the metrics files; no number in it is typed by hand.

In [ ]:
from IPython.display import Markdown, display
run("models/make_comparison.py")
display(Markdown((RESULTS / "comparison.md").read_text(encoding="utf-8")))

## Step 8 · Look at the results

**Table:** one row per model. *MAE* = average miss in years (lower is better) · *AUC* = how well the gap separates dementia from healthy people of similar age (0.5 = coin flip, 1 = perfect) · *ICC / SDC* = how consistent two scans of the same person are · *coverage* = how often the 95% range contains the true age.

In [ ]:
rows = []
for name in ("baseline_4mm", "cnn_2mm", "sfcn_ft", "volume", "ensemble", "ensemble_with_volume"):
    f = RESULTS / f"metrics_{name}.json"
    if not f.exists():
        continue
    m = json.loads(f.read_text())
    # ranges: calibrated version (cal_<model>) for single models; ensembles are calibrated already
    c = RESULTS / (f"metrics_{name}.json" if name.startswith("ensemble") else f"metrics_cal_{name}.json")
    cm = json.loads(c.read_text()) if c.exists() else {}
    cv, tr = m["clinical_validity"], m["test_retest"]
    rows.append({"model": name, "MAE (yrs)": m["mae"], "r": m["r"], "AUC": cv["auc"],
                 "AUC 95% CI": f"{cv['auc_ci'][0]:.2f}–{cv['auc_ci'][1]:.2f}", "ICC": tr["icc_3_1"],
                 "SDC (yrs)": tr["smallest_detectable_change"], "coverage": cm.get("interval_coverage_95"),
                 "range width (yrs)": cm.get("mean_interval_width")})
pd.DataFrame(rows).set_index("model").round(3)

**Charts:** guessed age vs real age for healthy people (each dot is one person, scored by a model that never saw them; the dashed line is a perfect guess), and the average miss by age decade.

In [ ]:
import matplotlib.pyplot as plt

models = [m for m in ("baseline_4mm", "cnn_2mm", "sfcn_ft", "volume", "ensemble") if (RESULTS / f"oof_{m}.csv").exists()]
fig, axes = plt.subplots(1, len(models), figsize=(4.2 * len(models), 4), sharex=True, sharey=True, squeeze=False)
for ax, m in zip(axes[0], models):
    d = pd.read_csv(RESULTS / f"oof_{m}.csv")
    n = d[(d["cdr"].fillna(0) == 0) & d["scan_id"].str.endswith("MR1") & (d["fold"] >= 0)]
    ax.scatter(n["age"], n["pred_age"], s=10, alpha=0.6)
    ax.plot([15, 100], [15, 100], "k--", lw=1)
    ax.set_title(f"{m}\nMAE {abs(n['pred_age'] - n['age']).mean():.2f} yrs")
    ax.set_xlabel("real age")
axes[0][0].set_ylabel("guessed brain age")
plt.tight_layout(); plt.show()

by_decade = pd.DataFrame({m: json.loads((RESULTS / f"metrics_{m}.json").read_text())["mae_by_age_decade"] for m in models})
by_decade.index = [f"{int(i)}s" for i in by_decade.index]
by_decade.plot.bar(figsize=(9, 3.5), ylabel="average miss (years)", title="Error by age decade"); plt.show()

**Confidence labels:** do "high confidence" guesses really miss by less? (healthy people, out-of-fold)

In [ ]:
for name in ["ensemble"] + [f"cal_{m}" for m in ("baseline_4mm", "cnn_2mm", "sfcn_ft")]:
    f = RESULTS / f"{name}_config.json"
    if f.exists():
        c = json.loads(f.read_text())
        print(f"{name:18s} rules: high < {c['rules']['high_width']} yrs wide, low > {c['rules']['low_width']} yrs")
        print("   ", {k: f"n={c['confidence_counts_normal'][k]}, MAE {v:.2f}" for k, v in c["mae_by_confidence_normal"].items()})

## Step 9 · Tests

All original app tests plus the new ones (scorer, ensemble, MRI API). The MRI API tests need the ensemble from Step 6.

In [ ]:
run("-m", "pytest", "-q", "-p", "no:cacheprovider")

## Step 10 · Use it in the app

In a PowerShell terminal:
```powershell
cd C:\brain-mri\brain
$env:PYTHONUTF8 = "1"
$env:NEUROTRAIL_MRI_MODEL = "ensemble"   # or baseline_4mm / cnn_2mm / sfcn_ft
C:\brain-mri\.venv\Scripts\python.exe web/server.py   # then open http://localhost:8600 → My data → MRI
```
If anything breaks during a demo, set `NEUROTRAIL_MRI_MODEL = "baseline_4mm"` to go back to the original model.

**Limits to state:** 316 healthy training people from one dataset (OASIS-1); scans must be preprocessed like the training scans (skull-stripped, FSL MNI152 2 mm); repeat-scan reliability comes from 20 young adults only; a research tool, not a diagnosis.